<a href="https://colab.research.google.com/github/ripims1/COMP-3608-PROJECT/blob/tyler-data-blending/Model_Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Global library declaration
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import lightgbm as lgb

In [ ]:
# ==============
# Neural Network
# ==============

#Load the data
train_df = pd.read_csv('train_processed.csv')
test_df = pd.read_csv('test_processed.csv')

#Separate Features (X) and Target (y)
X_train_raw = train_df.drop('Churn', axis=1).values
y_train_raw = train_df['Churn'].values
X_test_raw = test_df.values

#Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

#Convert to Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_raw, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

print(f"X_train shape: {X_train_tensor.shape}")
print(f"y_train shape: {y_train_tensor.shape}")

class ChurnModel(nn.Module):
    def __init__(self, input_size):
        super(ChurnModel, self).__init__()

        #Layer 1: Input (45 features) -> 64 Hidden Neurons
        self.fc1 = nn.Linear(input_size, 64)

        #Layer 2: 64 Neurons -> 32 Hidden Neurons
        self.fc2 = nn.Linear(64, 32)

        #Layer 3: 32 Neurons -> 1 Output
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        #Pass through Layer 1 and apply ReLU activation
        x = F.relu(self.fc1(x))

        #Pass through Layer 2 and apply ReLU activation
        x = F.relu(self.fc2(x))

        #Output Layer: Use Sigmoid to get a value between 0 and 1
        x = torch.sigmoid(self.fc3(x))

        return x

#Initialize the model using the number of columns (45)
input_dim = X_train_tensor.shape[1]
model = ChurnModel(input_dim)

print(model)

#Define the Loss Function
criterion = nn.BCELoss()

#Define the Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function and Optimizer are ready.")

#Combine X and y into a single Dataset object
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

#Create the DataLoader
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)

print("DataLoader is ready.")

epochs = 10

for epoch in range(epochs):
    model.train() #Set model to training mode
    running_loss = 0.0

    for inputs, labels in train_loader:
        #Clear the old gradients
        optimizer.zero_grad()

        #Forward Pass: Get predictions
        outputs = model(inputs)

        #Calculate Loss
        loss = criterion(outputs, labels)

        #Backward Pass
        loss.backward()

        #Optimizer Step: Update the weights
        optimizer.step()

        running_loss += loss.item()

    #Print progress after each epoch
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

print("Training Complete!")

#Switch to evaluation mode
model.eval()

#No need to track gradients
with torch.no_grad():
    #Pass the entire test set through the model
    test_outputs = model(X_test_tensor)

    #Convert back to a simple NumPy array for easier reading
    probabilities = test_outputs.numpy()

#Look at the first 5 predictions
print("First 5 Churn Probabilities:")
print(probabilities[:5])

#Get the starting ID
start_id = 594194

#Flatten the predictions
probs_flat = probabilities.flatten()

# Value for blending
prediction_NN = probs_flat

#Create the IDs
test_ids = range(start_id, start_id + len(probs_flat))

#Create the Final DataFrame
submission_df = pd.DataFrame({
    'id': test_ids,
    'Churn': probs_flat
})

In [13]:
# ==========
#  LightGBM
# ==========

# Use same scaled datasets as in the NN model
X_train_lgb = X_train_scaled
Y_train_lgb = y_train_raw
X_test_lgb = X_test_scaled

# Convert the training data to the LightGBM datset format
train_data = lgb.Dataset(X_train_lgb, label=Y_train_lgb)

# Define the model parametrs for LightGBM to control how it learns
params = {
    'objective': 'binary',        # Binary Classification problem
    'metric': 'auc',              # Area under ROC curve
    'boosting_type': 'gbdt',      # Gradient Boosting decision trees
    'learning_rate': 0.03,        # Learning step size
    'num_leaves': 64,             # Controls complexity of trees
    'feature_fraction': 0.8,      # Randomly select features to prevent overfitting
    'bagging_fraction': 0.8,      # Randomly sample data
    'bagging_freq': 5,            # Frequency of bagging
    'verbosity': -1               # Suppress training output
}

# Train the model using the dataset
lgb_model = lgb.train(params, train_data, num_boost_round=1000)

# Generate probability predictions for the test dataset and output between 0 and 1
prediction_LGBM = lgb_model.predict(X_test_lgb)

print("Successfully generated prediction")

In [14]:
# ==============
# Model Blending
# ==============

# Blends predictions in an 80 20 split due to performance difference
final_prediction = 0.85 * prediction_LGBM + 0.15 * prediction_NN

In [15]:
# ==================
#  FINAL SUBMISSION
# ==================

start_id = 594194
end_id = 848848

submission_df = pd.DataFrame({
    'id': range(start_id, end_id + 1),
    'Churn': final_prediction
})

submission_df.to_csv('submission4.csv', index=False)

print("Blended submission file created!")

Blended submission file created!
